# Logo Detection — YOLO Model Training & Comparison

This notebook trains and compares multiple YOLO model variants for logo detection.
The task is to detect 4 logo types (NYC, BP, SK, YORP), each labeled as either **Correct** or **Incorrect** placement — 8 classes total.

**Pipeline overview:**
1. Setup & dependencies
2. Dataset configuration and integrity checks
3. Baseline model (YOLOv8n — fast, low-memory sanity check)
4. Hyperparameter tuning (on YOLOv8s)
5. Tuned model training across multiple YOLO architectures for comparison
6. Evaluation and model comparison

## 1. Setup

In [ ]:
# !pip install ultralytics -q

In [2]:
import pandas as pd
import torch
from ultralytics import YOLO
import os
import yaml
from collections import Counter

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050


In [ ]:
# Uncomment if running on Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# path = "/content/drive/MyDrive/pubmat-checker"

## 2. Dataset Configuration

We define 8 classes (4 logo brands × 2 correctness states) and write a `data.yaml` file
that YOLO uses to locate images and labels during training.

Expected directory layout:
```
data/
  images/
    train/  val/  test/
  labels/
    train/  val/  test/
```

In [3]:
data = {
    "path": "./data",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "NYC_Correct",
        1: "NYC_Incorrect",
        2: "BP_Correct",
        3: "BP_Incorrect",
        4: "SK_Correct",
        5: "SK_Incorrect",
        6: "YORP_Correct",
        7: "YORP_Incorrect",
    }
}

with open("./data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml created.")

data.yaml created.


### 2.1 Data Integrity Check

Before training, we verify:
- Every image has a corresponding label file (and vice versa)
- Class distribution is visible across all splits

In [4]:
def check_split(base_path, class_names, split):
    """Check image/label integrity and print class distribution for a given split."""
    image_dir = os.path.join(base_path, "images", split)
    label_dir = os.path.join(base_path, "labels", split)

    image_stems = {
        os.path.splitext(f)[0]
        for f in os.listdir(image_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    }
    label_stems = {
        os.path.splitext(f)[0]
        for f in os.listdir(label_dir)
        if f.lower().endswith(".txt")
    }

    errors = []
    for stem in image_stems - label_stems:
        errors.append(f"Image has no label: {stem}")
    for stem in label_stems - image_stems:
        errors.append(f"Label has no image: {stem}")

    class_counts = Counter()
    for stem in label_stems:
        with open(os.path.join(label_dir, stem + ".txt")) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1

    total = sum(class_counts.values())
    print(f"\n--- {split.upper()} Split ---")
    print(f"Images: {len(image_stems)} | Labels: {len(label_stems)} | Annotations: {total}")
    print("\nClass Distribution:")
    rows = []
    for class_id, count in sorted(class_counts.items()):
        name = class_names.get(class_id, f"Unknown {class_id}")
        pct = (count / total) * 100 if total else 0
        print(f"  {name}: {count} ({pct:.1f}%)")
        rows.append({"Split": split, "Class": name, "Count": count, "Pct": round(pct, 1)})

    if errors:
        print("\nErrors:")
        for e in errors:
            print(" -", e)
    else:
        print("\nIntegrity check passed — no mismatches found.")

    return rows


with open("./data.yaml") as f:
    data_config = yaml.safe_load(f)

base_path = data_config["path"]
class_names = data_config["names"]
all_rows = []

for split in ["train", "val", "test"]:
    all_rows.extend(check_split(base_path, class_names, split))



--- TRAIN Split ---
Images: 1445 | Labels: 1445 | Annotations: 4195

Class Distribution:
  NYC_Correct: 596 (14.2%)
  NYC_Incorrect: 517 (12.3%)
  BP_Correct: 479 (11.4%)
  BP_Incorrect: 629 (15.0%)
  SK_Correct: 597 (14.2%)
  SK_Incorrect: 396 (9.4%)
  YORP_Correct: 609 (14.5%)
  YORP_Incorrect: 372 (8.9%)

Integrity check passed — no mismatches found.

--- VAL Split ---
Images: 334 | Labels: 334 | Annotations: 992

Class Distribution:
  NYC_Correct: 150 (15.1%)
  NYC_Incorrect: 111 (11.2%)
  BP_Correct: 128 (12.9%)
  BP_Incorrect: 141 (14.2%)
  SK_Correct: 140 (14.1%)
  SK_Incorrect: 87 (8.8%)
  YORP_Correct: 147 (14.8%)
  YORP_Incorrect: 88 (8.9%)

Integrity check passed — no mismatches found.

--- TEST Split ---
Images: 320 | Labels: 320 | Annotations: 923

Class Distribution:
  NYC_Correct: 133 (14.4%)
  NYC_Incorrect: 112 (12.1%)
  BP_Correct: 112 (12.1%)
  BP_Incorrect: 131 (14.2%)
  SK_Correct: 129 (14.0%)
  SK_Incorrect: 96 (10.4%)
  YORP_Correct: 124 (13.4%)
  YORP_Incorrect

## 3. Baseline Model: YOLOv8n

We start with **YOLOv8 Nano** (`yolov8n`) — the smallest variant — as a fast, low-memory baseline.
The goal here is not peak performance, but a quick sanity check that the data pipeline works
and to establish a minimum benchmark.

**Key baseline decisions:**
- `fliplr=0.0`, `flipud=0.0` — logos are orientation-sensitive; flipping would corrupt labels
- `mixup=0.0` — mixing correct/incorrect class samples would create ambiguous training signal
- `lrf=0.1` — final LR = `lr0 × lrf = 0.003 × 0.1 = 0.0003`, a gentler decay than the previous 0.01
- `cache=True` — speeds up repeated epoch reads

> **Note:** This uses `yolov8n`. The tuned models below use `yolov8s` (small) for more capacity.

In [5]:
model = YOLO("yolov8n.pt")

model.train(
    data="./data.yaml",
    name="baseline_yolov8n",
    epochs=80,
    imgsz=640,
    batch=16,
    seed=42,

    optimizer="AdamW",
    lr0=0.003,
    lrf=0.1,            
    weight_decay=0.0005,
    warmup_epochs=3.0,

    patience=20,       
    device=0,
    workers=2,
    cache=True,
    amp=True,
    val=True,

    degrees=0.0,
    translate=0.05,
    scale=0.8,
    fliplr=0.0,         
    flipud=0.0,
    mosaic=1.0,
    mixup=0.0           
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yol

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025A18793380>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [3]:
baseline_yolov8n = YOLO("runs/detect/baseline_yolov8n/weights/best.pt")

## 4. Hyperparameter Tuning

We use Ultralytics' built-in evolutionary tuner (`model.tune()`) to search for better hyperparameters.

### 4.1 Tuning YOLOv8

In [ ]:
model = YOLO("yolov8s.pt")

model.tune(
    data="./data.yaml",
    iterations=20,       
    imgsz=416,           
    batch=8,             
    epochs=15,           
    patience=5,
    optimizer="AdamW",
    plots=False,         
    save=False,          
    cache=True,
    workers=2,
    amp=True,
    name="tune_v8s"            
)

Tuner: Initialized Tuner instance with 'tune_dir=C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune2'
Tuner:  Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/20 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Saved C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune2\tune_scatter_plots.png
Saved C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune2\tune_fitness.png

Tuner: 1/20 iterations complete  (334.26s)
Tuner: Results saved to C:\Users\user\Documents\D

### 4.2 Tuning YOLOv11

In [11]:
model = YOLO("yolo11s.pt")

model.tune(
    data="./data.yaml",
    iterations=20,
    imgsz=416,
    batch=8,
    epochs=15,
    patience=5,
    optimizer="AdamW", 
    plots=False,
    save=False,
    cache=True,
    workers=2,
    amp=True,
    name="tune_v11s"
)

Tuner: Initialized Tuner instance with 'tune_dir=C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune3'
Tuner:  Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/20 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Saved C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune3\tune_scatter_plots.png
Saved C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune3\tune_fitness.png

Tuner: 1/20 iterations complete  (362.26s)
Tuner: Results saved to C:\Users\user\Documents\D

### 4.3 Tuning YOLOv26

In [12]:
model = YOLO("yolo26s.pt")

model.tune(
    data="./data.yaml",
    iterations=20,
    imgsz=416,
    batch=8,
    epochs=15,
    patience=5,
    optimizer="auto",  
    plots=False,
    save=False,
    cache=True,
    workers=2,
    amp=True,
    name="tune_v26s"
)

Tuner: Initialized Tuner instance with 'tune_dir=C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune_v26s'
Tuner:  Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/20 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Saved C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune_v26s\tune_scatter_plots.png
Saved C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\tune_v26s\tune_fitness.png

Tuner: 1/20 iterations complete  (450.70s)
Tuner: Results saved to C:\Users\user

## 5. Full Training

### 5.1 Hyperparameters

Define once and reuse across all training runs.

In [15]:
# v8 tuning results
with open("runs/detect/tune_v8s/best_hyperparameters.yaml") as f:
    best_hps = yaml.safe_load(f)

# v11 tuning results
with open("runs/detect/tune_v11s/best_hyperparameters.yaml") as f:
    best_hps_v11 = yaml.safe_load(f)

# v26 tuning results
with open("runs/detect/tune_v26s/best_hyperparameters.yaml") as f:
    best_hps_v26 = yaml.safe_load(f)

In [16]:
# Fixed settings that tuning doesn't control — always override
FIXED = dict(
    data="./data.yaml",
    epochs=150,
    patience=20,
    imgsz=640,
    batch=16,
    seed=42,
    device=0,
    workers=4,
    cache="disk",
    amp=True,
    val=True,
    save_period=10,
    # Logo-safe augmentation overrides
    fliplr=0.0,
    flipud=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    cls_pw=1.0,
)

# Tuned params as base, fixed settings take priority
TUNED_HPS = {**best_hps, **FIXED}
TUNED_HPS_V11 = {**best_hps_v11, **FIXED}
TUNED_HPS_V26 = {**best_hps_v26, **FIXED}

### 5.2 YOLOv8s Tuned 
Primary tuned model, full 150-epoch run.

In [6]:
model = YOLO("yolov8s.pt")
model.train(**TUNED_HPS, name="tuned_yolov8s")

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.48753, cache=disk, cfg=None, classes=None, close_mosaic=9, cls=0.49308, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.50502, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.00983, hsv_s=0.59554, hsv_v=0.40887, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00429, lrf=0.0155, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.91496, mosaic=0.97172, m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002BA969991D0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [7]:
tuned_yolov8s = YOLO("runs/detect/tuned_yolov8s/weights/best.pt")

### 5.4 YOLOv8m — Medium

Larger backbone (~26M params vs ~11M for small). May offer better precision on complex logo appearances,
but requires more memory and is slower to train and infer.

In [ ]:
model = YOLO("yolov8m.pt")
model.train(**TUNED_HPS, name="tuned_yolov8m")

In [27]:
model = YOLO("runs/detect/tuned_yolov8m/weights/last.pt")

model.train(resume=True)

New https://pypi.org/project/ultralytics/8.4.49 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.48753, cache=disk, cfg=None, classes=None, close_mosaic=9, cls=0.49308, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.50502, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.00983, hsv_s=0.59554, hsv_v=0.40887, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00429, lrf=0.0155, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\tuned_yolov8m\weights\last.pt, moment

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002BAA1AD4050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [28]:
tuned_yolov8m = YOLO("runs/detect/tuned_yolov8m/weights/best.pt")

### 5.5 YOLOv11s

YOLOv11 is an update in the Ultralytics family (released late 2024).
It uses an updated C3k2 backbone and SPPF improvements, offering better accuracy per parameter
compared to YOLOv8 at similar model sizes.

In [17]:
model = YOLO("yolo11s.pt")
model.train(**TUNED_HPS_V11, name="tuned_yolo11s")

New https://pypi.org/project/ultralytics/8.4.49 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tuned_yolo1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002BAB0D26190>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [18]:
tuned_yolo11s = YOLO("runs/detect/tuned_yolo11s/weights/best.pt")

### 5.6 YOLOv26s

Latest YOLO model

In [19]:
model = YOLO("yolo26s.pt")
model.train(**TUNED_HPS_V26, name="tuned_yolo26s")

New https://pypi.org/project/ultralytics/8.4.49 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00443, box=5.78288, cache=disk, cfg=None, classes=None, close_mosaic=8, cls=0.42553, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.01023, deterministic=True, device=0, dfl=1.43178, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01552, hsv_s=0.71166, hsv_v=0.49503, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00943, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.98, mosaic=0.98824

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002BAA1610050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

In [20]:
tuned_yolo26s = YOLO("runs/detect/tuned_yolo26s/weights/best.pt")

## 6. Evaluation

We evaluate all models on both the **validation** and **test** sets.
- **Val set** — used during training for early stopping; results here may be slightly optimistic
- **Test set** — held-out; the true measure of generalisation

Metrics collected: `mAP50`, `mAP50-95`, `Precision`, `Recall`.

In [21]:
os.makedirs("./results", exist_ok=True)

def get_metrics(model, name, split):
    """Evaluate a model on a given split and return overall + per-class DataFrames."""
    metrics = model.val(
        data="./data.yaml",
        plots=True,
        save=True,
        save_json=True,
        project="./results",
        name=f"{name}_{split}",
        split=split,
        verbose=False
    )

    overall_df = pd.DataFrame({
        "mAP50":     [metrics.box.map50],
        "mAP50-95":  [metrics.box.map],
        "Precision": [metrics.box.mp],
        "Recall":    [metrics.box.mr]
    })

    per_class_rows = []
    for i, class_name in metrics.names.items():
        p, r, ap50, ap = metrics.box.class_result(i)
        per_class_rows.append({
            "Class": class_name,
            "Precision": p,
            "Recall": r,
            "mAP50": ap50,
            "mAP50-95": ap
        })

    return overall_df, pd.DataFrame(per_class_rows)

### 6.1 Evaluate on Test Set

In [29]:
# Map of display name  loaded model
models = {
    "Baseline YOLOv8n":      baseline_yolov8n,
    "Tuned YOLOv8s":         tuned_yolov8s,
    "Tuned YOLOv8m":         tuned_yolov8m,
    "Tuned YOLOv11s":        tuned_yolo11s,
    "Tuned YOLOv26s":        tuned_yolo26s,
}

test_overall = {}
test_per_class = {}

for name, mdl in models.items():
    safe_name = name.lower().replace(" ", "_")
    overall, per_class = get_metrics(mdl, safe_name, "test")
    test_overall[name] = overall
    test_per_class[name] = per_class
    print(f"Done: {name}")

Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 218.542.3 MB/s, size: 175.1 KB)
val: Scanning C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\data\labels\test.cache... 320 images, 30 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 320/320 167.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 4.7it/s 4.2s0.2s
                   all        320        923      0.943      0.964       0.98      0.933
Speed: 3.9ms preprocess, 4.5ms inference, 0.0ms loss, 0.7ms postprocess per image
Saving C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\results\baseline_yolov8n_test2\predictions.json...
Results saved to C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\results\baseline_yolov8n_test2
Done: Baseline YOLOv8n
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA

### 6.2 Evaluate on Validation Set

In [30]:
val_overall = {}
val_per_class = {}

for name, mdl in models.items():
    safe_name = name.lower().replace(" ", "_")
    overall, per_class = get_metrics(mdl, safe_name, "val")
    val_overall[name] = overall
    val_per_class[name] = per_class
    print(f"Done: {name}")

Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)


val: Fast image access  (ping: 0.10.0 ms, read: 343.746.6 MB/s, size: 304.0 KB)
val: Scanning C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\data\labels\val.cache... 334 images, 27 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 334/334 100.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 5.8it/s 3.6s0.2s
                   all        334        992      0.964      0.949      0.981      0.937
Speed: 1.6ms preprocess, 4.4ms inference, 0.0ms loss, 1.2ms postprocess per image
Saving C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\results\baseline_yolov8n_val2\predictions.json...
Results saved to C:\Users\user\Documents\DARLYN\SMARTech\pubmat-checker\training\runs\detect\results\baseline_yolov8n_val2
Done: Baseline YOLOv8n
Ultralytics 8.4.38  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 6143MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 330.3307.6 M

### 6.3 Model Comparison Summary

Overall metrics for all models side-by-side.


In [31]:
def build_summary(overall_dict, split_label):
    df = pd.concat(overall_dict.values())
    df.index = overall_dict.keys()
    df["Split"] = split_label
    return df

summary_test = build_summary(test_overall, "Test")
summary_val  = build_summary(val_overall,  "Val")

combined = (
    pd.concat([summary_test, summary_val])
    .reset_index()
    .rename(columns={"index": "Model"})
    .set_index(["Model", "Split"])
    .round(4)
    .sort_index()
)


styled_combined = (
    combined.style
    .highlight_max(axis=0, color="#90EE90")
    .background_gradient(cmap="YlGn", axis=0)
    .format("{:.4f}")
)

styled_combined

styled_combined

In [ ]:

combined.to_csv("./results/model_comparison.csv")
print("Saved to ./results/model_comparison.csv")

### 6.4 Per-Class Breakdown

Inspect per-class performance for the best-performing model on the test set.
Low recall on a class may indicate insufficient training samples or ambiguous annotations.
Low precision may indicate visual confusion between classes (e.g. Correct vs Incorrect for the same brand).


In [25]:
best_model_name = "Tuned YOLOv8m"  

print(f"Per-Class Metrics — {best_model_name} (Test Set):")
test_per_class[best_model_name].set_index("Class").round(4)

Per-Class Metrics — Tuned YOLOv8m (Test Set):


,Precision,Recall,mAP50,mAP50-95
Class,,,,
NYC_Correct,0.9797,1.0000,0.9881,0.9881
NYC_Incorrect,0.9924,0.9911,0.9950,0.9794
BP_Correct,0.9248,1.0000,0.9851,0.9851
BP_Incorrect,0.9921,0.9600,0.9919,0.9698
SK_Correct,0.9517,1.0000,0.9905,0.9905
SK_Incorrect,0.9893,0.9627,0.9808,0.9392
YORP_Correct,0.9736,1.0000,0.9942,0.9941
YORP_Incorrect,0.9767,0.9739,0.9838,0.9153


In [26]:
print(f"Per-Class Metrics — {best_model_name} (Val Set):")
val_per_class[best_model_name].set_index("Class").round(4)

Per-Class Metrics — Tuned YOLOv8m (Val Set):


,Precision,Recall,mAP50,mAP50-95
Class,,,,
NYC_Correct,0.9868,0.9990,0.9949,0.9927
NYC_Incorrect,0.9982,0.9910,0.9950,0.9761
BP_Correct,0.9734,0.9922,0.9941,0.9876
BP_Incorrect,1.0000,0.9735,0.9946,0.9735
SK_Correct,0.9598,0.9929,0.9891,0.9818
SK_Incorrect,0.9879,0.9408,0.9836,0.9384
YORP_Correct,0.9799,0.9935,0.9927,0.9896
YORP_Incorrect,0.9882,0.9556,0.9855,0.9191


### 6.5 Export All Per-Class Results

In [ ]:
for name, df in test_per_class.items():
    safe = name.lower().replace(" ", "_")
    df.to_csv(f"./results/{safe}_per_class_test.csv", index=False)

for name, df in val_per_class.items():
    safe = name.lower().replace(" ", "_")
    df.to_csv(f"./results/{safe}_per_class_val.csv", index=False)

print("All per-class CSVs saved to ./results/")